# Clasificación, incertidumbre y explicabilidad con PneumoniaMNIST

Este notebook recorre el mismo flujo que el Notebook 2: entrenamiento MONAI, discriminación y calibración, Monte Carlo Dropout, saliencia, Grad-CAM y sensibilidad por oclusión. Aquí la tarea es **binaria** (Normal/Neumonía); en el Notebook 2 es multiclase de etiqueta única (ocho clases BloodMNIST).

**Objetivos**
- Comparar discriminación y calibración con métricas y un reliability diagram.
- Estimar variabilidad predictiva con Monte Carlo Dropout.
- Inspeccionar saliencia, Grad-CAM y oclusión sin tratarlos como evidencia causal.

> **Uso educativo, no clínico.** PneumoniaMNIST es un benchmark preprocesado. La API no expone identificadores de paciente, así que no permite auditar la separación por paciente; no debe usarse para decisiones clínicas.

In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

missing_packages = [
    package for package, module in (("monai", "monai"), ("medmnist", "medmnist"))
    if importlib.util.find_spec(module) is None
]
if missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.metrics import ConfusionMatrixDisplay, balanced_accuracy_score, confusion_matrix, roc_auc_score, roc_curve
from medmnist import PneumoniaMNIST
from monai.data import DataLoader, CacheDataset
from monai.networks.nets import DenseNet121
from monai.transforms import (
    Compose,
    EnsureChannelFirstd,
    EnsureTyped,
    RandAffined,
    RandAdjustContrastd,
    RandGaussianNoised,
    ScaleIntensityd,
)
from monai.utils import set_determinism
from monai.visualize import GradCAM

set_determinism(seed=17)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("MONAI:", __import__("monai").__version__)
print("MedMNIST:", __import__("medmnist").__version__)
print("Dispositivo:", device)

## 1. Datos reales: PneumoniaMNIST

PneumoniaMNIST contiene radiografías pediátricas de tórax clasificadas como **Normal** o **Neumonía**. Conservamos los splits oficiales. Con CUDA, el notebook usa todo el split de entrenamiento; en CPU activa el submuestreo rápido automáticamente. `QUICK_RUN` puede cambiarse manualmente.

Las imágenes son un benchmark pequeño y preprocesado, no estudios DICOM. El dataset no expone IDs de paciente: usaremos sus particiones oficiales, pero no afirmaremos que la separación por paciente esté auditada.

### Augmentation y balanceo con MONAI/PyTorch

El preprocesamiento común convierte la imagen a un canal, escala intensidades y establece tipos. Solo en entrenamiento aplicamos transformaciones aleatorias de MONAI: `RandAffined` (rotación, traslación y escala leves), `RandAdjustContrastd` (contraste) y `RandGaussianNoised` (ruido). Se vuelven a muestrear al acceder a cada ejemplo.

`WeightedRandomSampler` aumenta la probabilidad de extraer ejemplos de la clase minoritaria. El balance es aproximado en cada época y se logra con reemplazo, por lo que algunos ejemplos pueden repetirse. Solo se aplica a train; validation y test mantienen la distribución original para evaluar el comportamiento con la prevalencia del dataset.

Son ejemplos didácticos, no una receta clínica. Evitamos el flip horizontal para no invertir marcadores de lateralidad.

In [ ]:
SEED = 17
QUICK_RUN = device.type != "cuda"
TRAIN_FRACTION = 0.25 if QUICK_RUN else 1.0
IMAGE_SIZE = 64
BATCH_SIZE = 32 if device.type == "cuda" else 16
DATA_DIR = Path(os.environ.get("MONAI_DATA_DIRECTORY", Path.home() / ".cache" / "monai"))
DATA_DIR.mkdir(parents=True, exist_ok=True)
set_determinism(seed=SEED)

train_raw = PneumoniaMNIST(split="train", root=str(DATA_DIR), download=True, size=IMAGE_SIZE, as_rgb=False)
valid_raw = PneumoniaMNIST(split="val", root=str(DATA_DIR), download=True, size=IMAGE_SIZE, as_rgb=False)
test_raw = PneumoniaMNIST(split="test", root=str(DATA_DIR), download=True, size=IMAGE_SIZE, as_rgb=False)

class_names = {0: "Normal", 1: "Neumonía"}


def stratified_indices(labels, fraction, seed):
    """Elige índices conservando aproximadamente la proporción de cada clase.

    `fraction` controla qué parte de cada clase se selecciona y `seed` hace
    reproducible la selección. Si fraction es 1, devuelve todos los índices.
    """
    labels = np.asarray(labels).reshape(-1)
    if fraction >= 1.0:
        return np.arange(len(labels))
    rng = np.random.default_rng(seed)
    selected = []
    for class_id in np.unique(labels):
        class_indices = np.flatnonzero(labels == class_id)
        count = max(1, int(round(len(class_indices) * fraction)))
        selected.extend(rng.choice(class_indices, count, replace=False).tolist())
    return np.asarray(sorted(selected), dtype=int)


def records_from_split(raw, indices=None):
    """Convierte un split de MedMNIST en registros con claves para MONAI.

    Si no se indican índices, incluye todos los ejemplos del split.
    """
    if indices is None:
        indices = np.arange(len(raw.imgs))
    labels = raw.labels.reshape(-1)
    return [{"image": raw.imgs[index], "label": int(labels[index])} for index in indices]


train_indices = stratified_indices(train_raw.labels, TRAIN_FRACTION, SEED)
train_records = records_from_split(train_raw, train_indices)
valid_records = records_from_split(valid_raw)
test_records = records_from_split(test_raw)

In [ ]:
# Este pipeline común mantiene validation y test sin transformaciones aleatorias.
image_transform = Compose([
    EnsureChannelFirstd(keys="image", channel_dim="no_channel"),
    ScaleIntensityd(keys="image"),
    EnsureTyped(keys="image", dtype=torch.float32),
    EnsureTyped(keys="label", dtype=torch.long),
])

# Transformaciones moderadas para mostrar augmentation 2D en radiografías.
train_augmentation = Compose([
    RandAffined(
        keys="image",
        prob=0.5,
        rotate_range=(0.10,),
        translate_range=(3, 3),
        scale_range=(0.05, 0.05),
        mode="bilinear",
        padding_mode="border",
    ),
    RandAdjustContrastd(keys="image", prob=0.3, gamma=(0.8, 1.2)),
    RandGaussianNoised(keys="image", prob=0.2, mean=0.0, std=0.02),
])
train_transform = Compose([image_transform, train_augmentation])

train_ds = CacheDataset(data=train_records, transform=train_transform)
valid_ds = CacheDataset(data=valid_records, transform=image_transform)
test_ds = CacheDataset(data=test_records, transform=image_transform)

# Cada ejemplo recibe peso inverso al número de ejemplos de su clase.
train_labels = np.asarray([record["label"] for record in train_records])
train_class_counts = np.bincount(train_labels, minlength=2)
sample_weights = 1.0 / train_class_counts[train_labels]
train_sampler = torch.utils.data.WeightedRandomSampler(
    weights=torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=len(train_records),
    replacement=True,
)

# El sampler elige ejemplos balanceados; shuffle no se usa junto con sampler.
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=train_sampler, num_workers=0)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

for split_name, raw, records in [
    ("train", train_raw, train_records),
    ("validation", valid_raw, valid_records),
    ("test", test_raw, test_records),
]:
    counts = np.bincount([record["label"] for record in records], minlength=2)
    print(f"{split_name}: usados={len(records)}/{len(raw.imgs)} | Normal={counts[0]} | Neumonía={counts[1]}")
print(f"QUICK_RUN={QUICK_RUN} | train fraction={TRAIN_FRACTION} | batch size={BATCH_SIZE}")
print("WeightedRandomSampler equilibra las clases solo durante las extracciones de train.")

In [ ]:
# Comparamos una imagen preprocesada con una muestra aleatoria aumentada.
demo_index = next(i for i, record in enumerate(train_records) if record["label"] == 1)
original_image = image_transform(train_records[demo_index])["image"][0]
augmented_image = train_ds[demo_index]["image"][0]
fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
axes[0].imshow(original_image, cmap="gray", vmin=0, vmax=1)
axes[0].set_title("Original preprocesada")
axes[1].imshow(augmented_image, cmap="gray", vmin=0, vmax=1)
axes[1].set_title("Train: augmentation aleatoria")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()
print("Tamaño de entrada MONAI:", tuple(train_ds[0]["image"].shape), "| dispositivo:", device)

## 2. Clasificador MONAI y evaluación intermedia

`DenseNet121` produce dos logits: Normal y Neumonía. Después de cada época evaluamos el split de validación, sin usar el test para seleccionar épocas. El primer gráfico muestra la **loss de entrenamiento y validación en función de la época**: una separación creciente entre ambas puede indicar sobreajuste.

También seguimos ROC-AUC (ordenamiento de positivos y negativos), balanced accuracy (promedio de sensibilidad y especificidad), sensibilidad y especificidad. Estas métricas describen aspectos distintos; se reportan junto con la loss porque una sola medida no caracteriza por completo el rendimiento. Las métricas de clasificación usan umbral 0.5.

En CUDA, el entrenamiento usa AMP (`autocast` en `float16` y `GradScaler`); en CPU queda desactivado. La evaluación y las visualizaciones explicativas continúan en precisión completa.

In [ ]:
model = DenseNet121(
    spatial_dims=2,
    in_channels=1,
    out_channels=2,
    init_features=16,
    dropout_prob=0.2,
).to(device)
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
NUM_EPOCHS = 2 if QUICK_RUN else 8

# AMP usa autocast en CUDA; GradScaler protege el entrenamiento en float16.
USE_AMP = device.type == "cuda"
grad_scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
print(f"AMP float16: {'activado' if USE_AMP else 'desactivado'}")


def evaluate_split(loader):
    """Evalúa el modelo en un split y devuelve loss y métricas de clasificación.

    No actualiza los pesos. Las predicciones positivas se definen con umbral 0.5.
    """
    # eval() desactiva comportamientos de entrenamiento como Dropout.
    model.eval()
    loss_sum = 0.0
    probabilities, labels = [], []
    # No necesitamos gradientes durante la evaluación: ahorra memoria y cómputo.
    with torch.no_grad():
        for batch in loader:
            images = batch["image"].to(device)
            targets = batch["label"].to(device)
            logits = model(images)
            loss_sum += criterion(logits, targets).item()
            # Softmax convierte los logits en probabilidades; la columna 1 es Neumonía.
            probabilities.append(logits.softmax(dim=1)[:, 1].cpu().numpy())
            labels.append(targets.cpu().numpy())
    probabilities = np.concatenate(probabilities)
    labels = np.concatenate(labels).astype(int)
    predictions = probabilities >= 0.5
    true_positive = np.sum((predictions == 1) & (labels == 1))
    true_negative = np.sum((predictions == 0) & (labels == 0))
    false_positive = np.sum((predictions == 1) & (labels == 0))
    false_negative = np.sum((predictions == 0) & (labels == 1))
    metrics = {
        "loss": loss_sum / len(loader),
        "roc_auc": roc_auc_score(labels, probabilities),
        "balanced_accuracy": balanced_accuracy_score(labels, predictions),
        "sensitivity": true_positive / max(true_positive + false_negative, 1),
        "specificity": true_negative / max(true_negative + false_positive, 1),
    }
    return metrics

In [ ]:
# Guardamos cada métrica por época para poder compararlas en el gráfico siguiente.
history = {name: [] for name in ("train_loss", "validation_loss", "roc_auc", "balanced_accuracy", "sensitivity", "specificity")}
for epoch in range(NUM_EPOCHS):
    model.train()  # Activa el comportamiento de entrenamiento, incluido Dropout.
    train_loss_sum = 0.0
    for batch in train_loader:
        images = batch["image"].to(device)
        labels = batch["label"].to(device)
        # PyTorch acumula gradientes; los reiniciamos antes de cada lote.
        optimizer.zero_grad(set_to_none=True)
        # En CPU AMP queda desactivado; en CUDA algunas operaciones usan float16.
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(images)
            loss = criterion(logits, labels)
        # GradScaler es un no-op en CPU y ayuda a evitar gradientes demasiado pequeños en float16.
        grad_scaler.scale(loss).backward()
        grad_scaler.step(optimizer)
        grad_scaler.update()
        train_loss_sum += loss.item()

    # Medimos el rendimiento tras la época, sin actualizar los pesos con validation.
    validation_metrics = evaluate_split(valid_loader)
    history["train_loss"].append(train_loss_sum / len(train_loader))
    history["validation_loss"].append(validation_metrics["loss"])
    for metric_name in ("roc_auc", "balanced_accuracy", "sensitivity", "specificity"):
        history[metric_name].append(validation_metrics[metric_name])
    print(
        f"Época {epoch + 1}/{NUM_EPOCHS} | "
        f"train loss={history['train_loss'][-1]:.4f} | "
        f"validation loss={validation_metrics['loss']:.4f} | "
        f"ROC-AUC={validation_metrics['roc_auc']:.3f} | "
        f"balanced accuracy={validation_metrics['balanced_accuracy']:.3f} | "
        f"sensibilidad={validation_metrics['sensitivity']:.3f} | "
        f"especificidad={validation_metrics['specificity']:.3f}"
    )

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
epochs = np.arange(1, NUM_EPOCHS + 1)
axes[0].plot(epochs, history["train_loss"], "o-", label="Train")
axes[0].plot(epochs, history["validation_loss"], "o-", label="Validation")
axes[0].set_xlabel("Época")
axes[0].set_ylabel("Cross-entropy loss")
axes[0].set_title("Pérdida por época")
axes[0].legend()
axes[0].grid(alpha=0.2)
for metric_name in ("roc_auc", "balanced_accuracy", "sensitivity", "specificity"):
    axes[1].plot(epochs, history[metric_name], "o-", label=metric_name)
axes[1].set_xlabel("Época")
axes[1].set_ylabel("Métrica en validation")
axes[1].set_ylim(0, 1)
axes[1].set_title("Evaluación intermedia")
axes[1].legend()
axes[1].grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 3. Discriminación y calibración

Calculamos ROC-AUC y balanced accuracy para la discriminación; Brier score, ECE y reliability diagram para la calibración. Usamos validación para el diagrama y reportamos discriminación en el test oficial. La celda siguiente grafica la curva ROC en test para visualizar el compromiso entre sensibilidad y tasa de falsos positivos al variar el umbral; su AUC resume la discriminación, no la calibración.

ECE depende del número y los límites de los intervalos, así que no debe interpretarse de forma aislada.

In [ ]:
def predict_probabilities(model, loader):
    """Predice las dos probabilidades de clase y reúne las etiquetas reales.

    Devuelve tensores con forma (n_ejemplos, 2) y (n_ejemplos,).
    """
    model.eval()
    probabilities, labels = [], []
    with torch.no_grad():
        for batch in loader:
            logits = model(batch["image"].to(device))
            probabilities.append(logits.softmax(dim=1).cpu())
            labels.append(batch["label"].cpu())
    return torch.cat(probabilities), torch.cat(labels)


valid_probs, valid_labels = predict_probabilities(model, valid_loader)
valid_positive_probs = valid_probs[:, 1].numpy()
valid_targets = valid_labels.numpy()
# Brier: error cuadrático medio entre P(Neumonía) y la etiqueta 0/1.
valid_brier = np.mean((valid_positive_probs - valid_targets) ** 2)


def expected_calibration_error(targets, probabilities, n_bins=10):
    """Compara confianza y frecuencia observada dentro de intervalos de probabilidad.

    Devuelve el ECE, un promedio ponderado por el tamaño de cada intervalo ocupado.
    """
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for bin_index, (lower, upper) in enumerate(zip(bin_edges[:-1], bin_edges[1:])):
        # El último intervalo incluye 1.0; los anteriores evitan contar bordes dos veces.
        if bin_index == n_bins - 1:
            selected = (probabilities >= lower) & (probabilities <= upper)
        else:
            selected = (probabilities >= lower) & (probabilities < upper)
        if selected.any():
            confidence = probabilities[selected].mean()
            frequency = targets[selected].mean()
            ece += selected.mean() * abs(confidence - frequency)
    return ece


valid_ece = expected_calibration_error(valid_targets, valid_positive_probs)
print(f"Validación | Brier binario: {valid_brier:.4f} | ECE (10 bins): {valid_ece:.4f}")

In [ ]:
test_probs, test_labels = predict_probabilities(model, test_loader)
test_targets = test_labels.numpy()
test_predictions = test_probs.argmax(dim=1).numpy()
test_positive_probs = test_probs[:, 1].numpy()
print(f"Test oficial | ROC-AUC: {roc_auc_score(test_targets, test_positive_probs):.4f}")
print(f"Test oficial | balanced accuracy: {balanced_accuracy_score(test_targets, test_predictions):.4f}")

bin_edges = np.linspace(0.0, 1.0, 11)
mean_confidence, observed_rate = [], []
for bin_index, (lower, upper) in enumerate(zip(bin_edges[:-1], bin_edges[1:])):
    if bin_index == 10 - 1:
        selected = (valid_positive_probs >= lower) & (valid_positive_probs <= upper)
    else:
        selected = (valid_positive_probs >= lower) & (valid_positive_probs < upper)
    if selected.any():
        mean_confidence.append(valid_positive_probs[selected].mean())
        observed_rate.append(valid_targets[selected].mean())

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([0, 1], [0, 1], "--", color="gray", label="Calibración perfecta")
ax.plot(mean_confidence, observed_rate, "o-", label="Validation")
ax.set_xlabel("Probabilidad predicha de Neumonía")
ax.set_ylabel("Frecuencia observada de Neumonía")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.grid(alpha=0.2)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Filas = clase real; columnas = predicción. El orden mantiene Normal y Neumonía.
test_confusion = confusion_matrix(test_targets, test_predictions, labels=[0, 1])
confusion_display = ConfusionMatrixDisplay(
    confusion_matrix=test_confusion,
    display_labels=[class_names[0], class_names[1]],
)
confusion_display.plot(cmap="Blues", values_format="d", colorbar=False)
confusion_display.ax_.set_title("Matriz de confusión | test oficial")
confusion_display.ax_.set_xlabel("Predicción del modelo")
confusion_display.ax_.set_ylabel("Clase real")
plt.tight_layout()
plt.show()

In [ ]:
# La curva ROC muestra sensibilidad frente a tasa de falsos positivos para todos los umbrales.
# El área bajo esta curva (AUC) resume la capacidad de ordenar positivos por encima de negativos.
fpr, tpr, roc_thresholds = roc_curve(test_targets, test_positive_probs)
roc_auc = roc_auc_score(test_targets, test_positive_probs)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, label=f"Modelo (AUC = {roc_auc:.3f})")
ax.plot([0, 1], [0, 1], "--", color="gray", label="Clasificador aleatorio")
ax.set_xlabel("Tasa de falsos positivos")
ax.set_ylabel("Sensibilidad (tasa de verdaderos positivos)")
ax.set_title("Curva ROC | test oficial")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.grid(alpha=0.2)
ax.legend()
plt.tight_layout()
plt.show()

## 4. Monte Carlo Dropout

Dejamos el modelo en modo evaluación para que las capas de normalización no cambien su comportamiento, pero activamos las capas Dropout. Comparamos radiografías originales con las mismas imágenes bajo ruido gaussiano leve. La segunda condición es una **corrupción controlada**, no una cohorte OOD clínica ni una garantía de incertidumbre.

In [ ]:
def mc_dropout_predictions(model, images, n_passes=20):
    """Genera varias predicciones manteniendo Dropout activo durante evaluación.

    Devuelve probabilidades con forma (pasadas, ejemplos, clases); su dispersión
    estima variabilidad entre pasadas, no una garantía de incertidumbre clínica.
    """
    model.eval()
    dropout_types = (torch.nn.Dropout, torch.nn.Dropout2d, torch.nn.Dropout3d)
    # Conservamos el modo de evaluación en el resto del modelo y activamos solo Dropout.
    for module in model.modules():
        if isinstance(module, dropout_types):
            module.train()
    with torch.no_grad():
        samples = torch.stack([model(images).softmax(dim=1) for _ in range(n_passes)])
    model.eval()
    return samples


mc_images = next(iter(test_loader))["image"][:24].to(device)
corrupted_images = (mc_images + torch.randn_like(mc_images) * 0.12).clamp(0, 1)
clean_samples = mc_dropout_predictions(model, mc_images)
corrupted_samples = mc_dropout_predictions(model, corrupted_images)

clean_std = clean_samples[:, :, 1].std(dim=0).cpu().numpy()
corrupted_std = corrupted_samples[:, :, 1].std(dim=0).cpu().numpy()
print(f"Desviación estándar media, test original: {clean_std.mean():.4f}")
print(f"Desviación estándar media, ruido controlado: {corrupted_std.mean():.4f}")
fig, ax = plt.subplots(figsize=(6, 4))
ax.boxplot([clean_std, corrupted_std], tick_labels=["Original", "Ruido"])
ax.set_ylabel("Desviación estándar de P(Neumonía)")
ax.set_title("Variación entre pasadas MC Dropout")
ax.grid(axis="y", alpha=0.2)
plt.show()

## 5. Saliencia: gradiente respecto de la imagen

El gradiente del score respecto de los píxeles aproxima cuánto cambia localmente la salida ante pequeñas perturbaciones. Para facilitar la comparación, mostraremos los mismos cinco ejemplos del test en las tres técnicas de explicación. En cada imagen se informa la clase real, la predicción y la probabilidad asignada a esa predicción.

La saliencia se calcula para la clase predicha por el modelo. El mapa muestra magnitudes normalizadas por imagen: permite comparar patrones espaciales, no la intensidad absoluta entre pacientes. No implica causalidad ni demuestra que el modelo haya localizado una lesión.

In [ ]:
# Fijamos los mismos cinco casos de test para poder compararlos en Grad-CAM y oclusión.
explain_examples = [test_ds[index] for index in range(min(5, len(test_ds)))]
model.eval()
fig, axes = plt.subplots(len(explain_examples), 2, figsize=(8, 3.2 * len(explain_examples)))

for row, sample in enumerate(explain_examples):
    # requires_grad permite pedir la derivada del score respecto de los píxeles de entrada.
    image_tensor = sample["image"].unsqueeze(0).to(device).detach().requires_grad_(True)
    true_label = int(sample["label"])
    model.zero_grad(set_to_none=True)
    logits = model(image_tensor)
    probabilities = logits.softmax(dim=1)[0]
    predicted_label = int(probabilities.argmax())
    predicted_probability = float(probabilities[predicted_label].detach().cpu())

    # La derivada del score de la clase predicha da una sensibilidad local de los píxeles.
    logits[0, predicted_label].backward()
    saliency = image_tensor.grad.detach().abs()[0, 0].cpu().numpy()
    # El valor absoluto muestra magnitud, y la normalización facilita ver cada mapa.
    saliency = saliency / (saliency.max() + 1e-8)
    image_for_plot = image_tensor.detach()[0, 0].cpu().numpy()
    outcome = "acierto" if predicted_label == true_label else "error"

    axes[row, 0].imshow(image_for_plot, cmap="gray")
    axes[row, 0].set_title(
        f"Real: {class_names[true_label]} | Modelo: {class_names[predicted_label]} "
        f"({predicted_probability:.2f}, {outcome})"
    )
    axes[row, 1].imshow(saliency, cmap="inferno", vmin=0, vmax=1)
    axes[row, 1].set_title("Saliencia normalizada |gradiente|")
    for ax in axes[row]:
        ax.axis("off")

fig.suptitle("Saliencia en cinco ejemplos del test", y=1.002)
plt.tight_layout()
plt.show()

## 6. Grad-CAM en distintas capas

Comparamos `features.denseblock1` a `features.denseblock4` para los mismos cinco ejemplos y la misma clase predicha en cada imagen. La primera columna muestra la radiografía y las siguientes, los mapas superpuestos de cada capa.

Las capas tempranas suelen conservar más detalle espacial; las profundas tienden a representar información más específica para la clase, pero con mapas de menor resolución. Grad-CAM es una atribución espacial gruesa del score, no una máscara de neumonía ni una explicación causal. Las zonas destacadas no prueban que sean hallazgos clínicos relevantes.

In [ ]:
gradcam_layers = [
    "features.denseblock1",
    "features.denseblock2",
    "features.denseblock3",
    "features.denseblock4",
]
model.eval()
gradcam_cases = []

# Guardamos una sola predicción por imagen para comparar todas las capas bajo la misma clase.
for sample in explain_examples:
    image = sample["image"].unsqueeze(0).to(device)
    true_label = int(sample["label"])
    with torch.no_grad():
        probabilities = model(image).softmax(dim=1)[0].cpu().numpy()
    predicted_label = int(probabilities.argmax())
    predicted_probability = float(probabilities[predicted_label])
    gradcam_cases.append((image, true_label, predicted_label, predicted_probability))

In [ ]:
fig, axes = plt.subplots(
    len(gradcam_cases), len(gradcam_layers) + 1,
    figsize=(16, 3.8 * len(gradcam_cases)),
)

for row, (image, true_label, predicted_label, predicted_probability) in enumerate(gradcam_cases):
    outcome = "acierto" if predicted_label == true_label else "error"
    axes[row, 0].imshow(image[0, 0].detach().cpu(), cmap="gray")
    axes[row, 0].set_title(
        f"Real: {class_names[true_label]} | Modelo: {class_names[predicted_label]} "
        f"({predicted_probability:.2f}, {outcome})"
    )
    axes[row, 0].axis("off")

# Cambiamos una capa objetivo cada vez y explicamos siempre el mismo score predicho.
for column, layer_name in enumerate(gradcam_layers, start=1):
    layer_gradcam = GradCAM(
        nn_module=model,
        target_layers=layer_name,
        postprocessing=lambda activation: activation,
    )
    for row, (image, _, predicted_label, _) in enumerate(gradcam_cases):
        cam = layer_gradcam(image, class_idx=predicted_label)[0, 0].detach().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        axes[row, column].imshow(image[0, 0].detach().cpu(), cmap="gray")
        axes[row, column].imshow(cam, cmap="jet", alpha=0.4, vmin=0, vmax=1)
        axes[row, column].axis("off")
    axes[0, column].set_title(f"Grad-CAM\n{layer_name}")
    del layer_gradcam

fig.suptitle("Grad-CAM: comparación de capas en cinco ejemplos del test", y=1.002)
plt.tight_layout()
plt.show()

## 7. Sensibilidad por oclusión

Para cada uno de los mismos cinco ejemplos, reemplazamos parches por cero y medimos el cambio en P(Neumonía). La figura muestra la región ocluida cuya retirada produce la mayor caída observada y el score antes/después; si el score sube, la oclusión tuvo el efecto contrario.

Con imágenes de 64×64 y parches de 16×16 con stride 16 se evalúan 16 oclusiones por imagen. El resultado depende del tamaño, stride y valor de reemplazo del parche: es una prueba de sensibilidad, no una intervención causal ni una localización de lesión.

In [ ]:
def occlusion_map(model, image, class_index=1, patch_size=16, stride=16):
    """Mide cómo cambia el score al ocultar parches de una imagen.

    Devuelve el mapa promedio de cambios, el score original y el score tras
    la oclusión con mayor caída. `patch_size` y `stride` controlan el barrido.
    """
    model.eval()
    height, width = image.shape[-2:]
    score_map = np.zeros((height, width), dtype=np.float32)
    counts = np.zeros_like(score_map)
    largest_score_drop = -np.inf
    with torch.no_grad():
        baseline = model(image).softmax(dim=1)[0, class_index].item()
        for top in range(0, height - patch_size + 1, stride):
            for left in range(0, width - patch_size + 1, stride):
                occluded = image.clone()
                occluded[:, :, top:top + patch_size, left:left + patch_size] = 0
                occluded_score = model(occluded).softmax(dim=1)[0, class_index].item()
                score_drop = baseline - occluded_score
                largest_score_drop = max(largest_score_drop, score_drop)
                score_map[top:top + patch_size, left:left + patch_size] += score_drop
                counts[top:top + patch_size, left:left + patch_size] += 1
    # Un píxel puede pertenecer a varios parches; promediamos sus contribuciones.
    return score_map / np.maximum(counts, 1), baseline, baseline - largest_score_drop

In [ ]:
model.eval()
fig, axes = plt.subplots(len(explain_examples), 2, figsize=(9, 3.8 * len(explain_examples)))

for row, sample in enumerate(explain_examples):
    image = sample["image"].unsqueeze(0).to(device)
    true_label = int(sample["label"])
    with torch.no_grad():
        probabilities = model(image).softmax(dim=1)[0].cpu().numpy()
    predicted_label = int(probabilities.argmax())
    predicted_probability = float(probabilities[predicted_label])

    # Se conserva P(Neumonía) para que todos los ejemplos midan el mismo score.
    occlusion, baseline_score, score_after_occlusion = occlusion_map(model, image)
    outcome = "acierto" if predicted_label == true_label else "error"

    axes[row, 0].imshow(image[0, 0].detach().cpu(), cmap="gray")
    axes[row, 0].set_title(
        f"Real: {class_names[true_label]} | Modelo: {class_names[predicted_label]} "
        f"({predicted_probability:.2f}, {outcome})"
    )
    axes[row, 1].imshow(image[0, 0].detach().cpu(), cmap="gray")
    axes[row, 1].imshow(occlusion, cmap="coolwarm", alpha=0.5)
    axes[row, 1].set_title(
        f"P(Neumonía): {baseline_score:.2f} → {score_after_occlusion:.2f} "
        "(parche con mayor caída)"
    )
    for ax in axes[row]:
        ax.axis("off")

fig.suptitle("Sensibilidad por oclusión en cinco ejemplos del test", y=1.002)
plt.tight_layout()
plt.show()

## Comparación y discusión

1. ¿Qué diferencia hay entre ROC-AUC y calibración en validation?
2. ¿La dispersión de MC Dropout cambia con el ruido controlado? ¿Qué no podemos concluir?
3. ¿Qué regiones resaltan saliencia y Grad-CAM? ¿Por qué no equivalen a una máscara ni prueban causalidad?
4. ¿Cómo cambia el score al ocluir distintas regiones y qué depende del tamaño del parche?
5. ¿Qué información adicional haría falta para evaluar generalización clínica y separación por paciente?

**Idea clave:** discriminación, calibración, incertidumbre y explicación responden preguntas distintas. Ninguna valida por sí sola un sistema clínico.